## NotMIWAE — Latent Space Extraction
Loads all saved `.pth` models and extracts `mu`, `logvar`, and `z_sample` for every dataset.
Outputs are saved to `/kaggle/working/latents/`.

In [ ]:
# ── Cell 1: Setup & encoder patch ────────────────────────────────────────────
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from notmiwae_pytorch import NotMIWAE
from notmiwae_pytorch.models.base import Encoder
import notmiwae_pytorch.models.notmiwae as notmiwae_module

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PREPROCESSED = '/kaggle/working/preprocessed'
MODELS_DIR   = '/kaggle/working'
OUTPUT_DIR   = '/kaggle/working/latents'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Must patch BEFORE any NotMIWAE() call — matches the architecture saved in .pth
class TwoLayerEncoder(Encoder):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__(input_dim, hidden_dim, latent_dim)
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        self.mu     = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

notmiwae_module.Encoder = TwoLayerEncoder
print('Encoder patched OK')

In [ ]:
# ── Cell 2: Dataset configs ───────────────────────────────────────────────────
# Each entry maps exactly to the hyperparams used during training.
# 'data_file' points to the version of the data the model was trained on.

CONFIGS = [
    {
        'name'       : 'mnist',
        'model_file' : 'mnist_notmiwae_model.pth',
        'data_file'  : 'mnist_missing.npz',
        'input_dim'  : 784,
        'latent_dim' : 50,
        'hidden_dim' : 128,
        'n_samples'  : 20,
    },
    {
        'name'       : 'fashion_mnist',
        'model_file' : 'fashion_notmiwae_model.pth',
        'data_file'  : 'fashion_mnist_missing.npz',
        'input_dim'  : 784,
        'latent_dim' : 50,
        'hidden_dim' : 128,
        'n_samples'  : 20,
    },
    {
        'name'       : 'usps',
        'model_file' : 'usps_notmiwae_model.pth',
        'data_file'  : 'usps_missing.npz',
        'input_dim'  : 256,
        'latent_dim' : 20,
        'hidden_dim' : 128,
        'n_samples'  : 20,
    },
    {
        'name'       : 'stl10',
        'model_file' : 'stl10_notmiwae_model.pth',
        'data_file'  : 'stl10_missing.npz',
        'input_dim'  : 2048,
        'latent_dim' : 50,
        'hidden_dim' : 256,
        'n_samples'  : 20,
    },
    {
        'name'       : 'reuters',
        'model_file' : 'reuters_notmiwae_model.pth',
        'data_file'  : 'reuters_missing.npz',
        'input_dim'  : 2000,
        'latent_dim' : 50,
        'hidden_dim' : 128,
        'n_samples'  : 20,
    },
    {
        'name'       : 'russian_housing',
        'model_file' : 'russian_housing_notmiwae_model.pth',
        'data_file'  : 'russian_housing.npz',  # trained on real NaN, no artificial mask
        'input_dim'  : 4,
        'latent_dim' : 2,
        'hidden_dim' : 32,
        'n_samples'  : 20,
    },
    {
        'name'       : 'horse_colic',
        'model_file' : 'horse_colic_notmiwae_model.pth',
        'data_file'  : 'horse_colic.npz',
        'input_dim'  : 7,
        'latent_dim' : 3,
        'hidden_dim' : 32,
        'n_samples'  : 20,
    },
    {
        'name'       : 'nhanes',
        'model_file' : 'nhanes_notmiwae_model.pth',
        'data_file'  : 'nhanes.npz',
        'input_dim'  : 2094,
        'latent_dim' : 50,
        'hidden_dim' : 256,
        'n_samples'  : 10,   # trained with n_samples=10
    },
    {
        'name'       : 'hhar',
        'model_file' : 'hhar_notmiwae_model.pth',
        'data_file'  : 'hhar_missing.npz',
        'input_dim'  : 18,
        'latent_dim' : 10,
        'hidden_dim' : 64,
        'n_samples'  : 20,
    },
]
print(f'{len(CONFIGS)} datasets configured')

In [ ]:
# ── Cell 3: Latent extraction helper ─────────────────────────────────────────

def extract_latents(model, X_filled_np, mask_np, batch_size=512):
    """
    Runs the encoder over the full dataset in batches.

    Returns
    -------
    mu       : (N, latent_dim)  — posterior mean
    logvar   : (N, latent_dim)  — posterior log-variance
    z_sample : (N, latent_dim)  — one reparameterised draw  mu + eps*std
    """
    model.eval()

    X_t    = torch.tensor(X_filled_np, dtype=torch.float32)
    mask_t = torch.tensor(mask_np,     dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t, mask_t),
                        batch_size=batch_size, shuffle=False)

    all_mu, all_logvar = [], []

    with torch.no_grad():
        for x_b, m_b in loader:
            x_b = x_b.to(device)
            m_b = m_b.to(device)
            # NotMIWAE encoder takes (x_observed, mask)
            # x_observed = x * mask  (zeros fill missing positions)
            mu_b, lv_b = model.encoder(x_b * m_b, m_b)
            all_mu.append(mu_b.cpu())
            all_logvar.append(lv_b.cpu())

    mu     = torch.cat(all_mu,     dim=0).numpy()
    logvar = torch.cat(all_logvar, dim=0).numpy()
    std      = np.exp(0.5 * logvar)
    z_sample = mu + np.random.randn(*mu.shape).astype(np.float32) * std

    return mu, logvar, z_sample

print('Helper defined')

In [ ]:
# ── Cell 4: Main loop — load each model and extract latents ──────────────────

summary = []

for cfg in CONFIGS:
    name       = cfg['name']
    model_path = os.path.join(MODELS_DIR,   cfg['model_file'])
    data_path  = os.path.join(PREPROCESSED, cfg['data_file'])

    if not os.path.exists(model_path):
        print(f'[SKIP] {name}: model file not found  →  {model_path}')
        continue
    if not os.path.exists(data_path):
        print(f'[SKIP] {name}: data file not found   →  {data_path}')
        continue

    print(f"\n{'='*55}")
    print(f'  {name}')
    print(f"{'='*55}")

    # ── load data ──────────────────────────────────────────
    raw      = np.load(data_path)
    X        = raw['X'].astype(np.float32)
    y        = raw['y'] if 'y' in raw else None

    mask_np  = np.isfinite(X).astype(np.float32)
    X_filled = np.nan_to_num(X, nan=0.0)

    print(f'  data shape : {X.shape}')
    print(f'  missing    : {(1 - mask_np).mean()*100:.2f}%')

    # ── rebuild model (must match saved architecture exactly) ─
    model = NotMIWAE(
        input_dim       = cfg['input_dim'],
        latent_dim      = cfg['latent_dim'],
        hidden_dim      = cfg['hidden_dim'],
        n_samples       = cfg['n_samples'],
        missing_process = 'selfmasking',
        out_dist        = 'gauss',
    ).to(device)

    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    print(f'  weights loaded from {cfg["model_file"]}')

    # ── extract ────────────────────────────────────────────
    mu, logvar, z_sample = extract_latents(model, X_filled, mask_np)

    kl = (-0.5 * (1 + logvar - mu**2 - np.exp(logvar))).sum(axis=1).mean()
    print(f'  mu shape   : {mu.shape}')
    print(f'  mu mean/std: {mu.mean():.4f} / {mu.std():.4f}')
    print(f'  mean KL    : {kl:.4f}')

    # ── save ───────────────────────────────────────────────
    out_path = os.path.join(OUTPUT_DIR, f'{name}_latents.npz')
    save_kw  = dict(mu=mu, logvar=logvar, z_sample=z_sample)
    if y is not None:
        save_kw['y'] = y
    np.savez_compressed(out_path, **save_kw)
    print(f'  saved → {out_path}')

    summary.append({'name': name, 'shape': mu.shape, 'path': out_path})

    del model
    torch.cuda.empty_cache()

print(f"\n{'='*55}")
print('ALL DONE')
print(f"{'='*55}")
for s in summary:
    print(f"  {s['name']:<22} mu: {str(s['shape']):<16} → {s['path']}")

In [ ]:
# ── Cell 5: Quick sanity check ────────────────────────────────────────────────
# Reload one saved file and print basic stats to confirm everything looks right.

for s in summary:
    d  = np.load(s['path'])
    mu = d['mu']
    print(f"{s['name']:<22}  mu {mu.shape}  "
          f"mean={mu.mean():+.3f}  std={mu.std():.3f}"
          + (f"  labels={np.unique(d['y'])}" if 'y' in d else ''))